Quick and dirty notebook for simulation of coherent scattering data of magnetic samples in fraunhofer far-field regime

# Import

In [ ]:
# Import general libraries
import os
from os.path import join, split
from importlib import reload
from copy import deepcopy
import numpy as np

# plotting
import matplotlib.pyplot as plt

%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Imports from our own codebase
from scattering_calculator.sample_generator import pattern_generator  # magnetic pattern generators
from scattering_calculator.interactive.interactive_widgets import cimshow  # interactive image viewer
from scattering_calculator.simulation_pipelines import simulation_configuration  # config dataclasses
import scattering_calculator.utils as utils  # I/O helpers (save_simulation_arrays_hdf5, etc.)
from fomocid import DATA_ROOT  # project root: one level above the fomocid repo

## Simulation pipeline overview

This notebook simulates coherent X-ray scattering (Fourier transform holography, FTH) from a magnetic thin-film sample. The pipeline runs in the following order:

1. **Experimental geometry** — define the X-ray source energy/polarisation and detector layout (pixel size, distance, beamstop).
2. **Sample** — parse the multilayer material recipe, generate a magnetic domain pattern, and punch the FTH holography mask (object hole + reference holes) into the aperture.
3. **Hologram computation** — propagate the beam through the sample using the Jones matrix formalism for both circular-right (CR) and circular-left (CL) polarisations, then project the exit wavefield onto the detector.
4. **Post-processing** — compute helicity difference/sum, FTH reconstructions, and frame averages.
5. **Export** — save arrays and experimental metadata to an HDF5 file.

### EXPERIMENTAL GEOMETRY

In [ ]:
# ===================
# X-ray Source
# ===================
metadata = {}  # accumulates scalar metadata from all config objects; saved alongside arrays in HDF5

#######################################
# Create config class for x-ray source
xrayconfig = simulation_configuration.XRayConfig(
    energy           = 787.9, # eV
    pol              = "CR",  # initial polarization (overridden per-helicity in the propagation loop)
    photon_flux      = 1e10, # Photons per second
    coherence_length = (20e-6, 20e-6),  # in m, (y, x); controls partial coherence blur on the detector
)
xrayconfig.setup()  # derives beam_params (wavelength, wavenumber, etc.) from energy

In [ ]:
# ==================
# DETECTOR-BEAMSTOP GEOMETRY
# ==================
detector_pixel_size = 20e-6  # in m
detector_pixel_shape = (1300, 1300)
detector_distance = 0.02  # in m
detector_center = (650, 650)  # in px

detector_params = {
    "readout_noise_average": 50,
    "noise_rms": 3,
    "detector_threshold": 64e3,
    "counts_per_photon": 100,
    "quantum_efficiency": 0.85,
}
measurement_config = {
    "number_frames": 1,
    "max_counts_per_image": None, #if None there is no renormalization
    "exposure_time": 5., # s
}
artifacts_config = {
    "counts_per_photon": 100,
    "sigma_photon": 0.75,
    "photon_n_classes": 1,
    "photon_n_variants": 30,
    "photon_kernel_size": 9,
    "photon_irregularity": 2.0,
    "regenerate_photon_kernels": True,
}

# Optional: Define a beamstop
# Basic parameters for beamstop
beamstop_distance = 0.001  # in m
beamstop_center = np.array(detector_pixel_shape) // 2  # in px; centred on the direct beam

# Select beamstop method and parameters
beamstop_method = "circular"  # "circular", "rectangular", None
beamstop_radius = 0.55e-3  # in m
beamstop_sigma = 1  # in m
beamstop_wire_width = 0.05e-3  # in m
beamstop_wire_bend = 0.1e-3  # in m
beamstop_theta=np.pi/6
beamstop_ellipticity_range = (0.9,1.1)
beamstop_roughness= 0.05
beamstop_roughness_modes = (3, 9)



####################################
# Create config class for beamstop
beamstop_config = simulation_configuration.BeamstopConfig(
    bs_method=beamstop_method,
    bs_detector_distance=beamstop_distance,
    bs_center=beamstop_center,
    bs_config={"radius"         : beamstop_radius,
               "angle"          : beamstop_theta,
               "sigma"          : beamstop_sigma,
               "ellipticity"    : beamstop_ellipticity_range,
               "roughness"      : beamstop_roughness,
               "rougness_modes" : beamstop_roughness_modes,
               "wire_width"     : beamstop_wire_width,
               "wire_bend"      : beamstop_wire_bend,
               "seed"           : None,
               },
)

# Create config class for detector
detectorconfig = simulation_configuration.DetectorConfig(
    pixel_size=detector_pixel_size,
    shape=detector_pixel_shape,
    sample_to_detector_distance=detector_distance,
    detector_center=detector_center,
    detector_params=detector_params,
    measurement_config=measurement_config,
    artifacts_config=artifacts_config,
    beamstop_config=beamstop_config
    
)
detectorconfig.setup()
detectorconfig.visualize_beamstop()  # sanity check: confirm beamstop placement before simulation

## This is the resolution we will have thanks to the detector
real_space_pixel_size = (
    detectorconfig.calc_realspace_resolution(xrayconfig.beam_params) / 2  # /2 for 2× oversampling
)
metadata.update(beamstop_config.get_metadata(prefix="beamstop/"))  # record beamstop params

`real_space_pixel_size` is derived from the detector geometry and photon wavelength via the Nyquist criterion: it sets the physical size of one pixel in the sample plane and determines the field of view of the simulation. Dividing by 2 oversamples by a factor of 2, which avoids aliasing in the far-field propagation.

# SAMPLE 

### SAMPLE STACK STRUCTURE AND OPTICAL PROPERTIES

In [ ]:
# ===================
# MATERIAL RECIPE
# ===================
recipe = "Au(700)/Cr(300)/SiN(200)/Co(90)/Pt(120)/Al(60)"  # layer stack in nm, top to bottom
oversampling = 2  # lateral oversampling factor: sample grid is 2× the detector grid to avoid wrap-around artefacts

sample_shape = np.array([
        0,  # Nz: sentinel — updated automatically by sampleconfig.setup() to match number of layers
        oversampling * detectorconfig.shape[0],
        oversampling * detectorconfig.shape[1],
    ],
    dtype=int,
)  # in pixels

sampleconfig = simulation_configuration.SampleConfig(
    recipe=recipe,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    xray_config=xrayconfig,
    sample_name="Test_Sample",
)
sampleconfig.setup()

`sample_shape[0] = 0` is a sentinel — it is updated automatically to match the number of layers after `setup()`. The lateral dimensions are set to twice the detector size so that the sample field of view covers the full detector without wrap-around artefacts during the Fourier propagation.

### Magnetic domain pattern

Choose **one** of the two cells below: wavy stripes or a skyrmion lattice. Run only that cell, then continue with the magnetization mapping.

In [ ]:
# --- Option A: wavy stripe pattern ---
stripe_width = 20e-9       # periodicity of the stripes in m
sigma = 1e-9               # edge smoothing (Gaussian blur) in m
angle_stripes = np.pi / 4  # orientation of stripe wavevector (radians from x-axis)
wave_amplitudes = 80e-9    # peak-to-peak lateral waviness amplitude in m
wave_scale = 20e-9         # spatial period of the waviness modulation in m

magnetic_pattern_config = simulation_configuration.MagneticPatternConfig(
    pattern_type_method="wavy_stripe_pattern",
    shape=sample_shape[1:],                  # 2-D lateral shape (Ny, Nx)
    real_space_pixel_size=real_space_pixel_size,
    pattern_config_length={                  # parameters given in metres
        "stripe_width": stripe_width,
        "sigma": sigma,
        "waviness_amplitude": wave_amplitudes,
        "waviness_scale": wave_scale,
    },
    pattern_config={
        "angle_stripes": angle_stripes,
    },
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_config.plot_pattern()

%%time
# --- Option B: skyrmion lattice ---
skyrmion_radius = 3e-9          # radius of a single skyrmion core in m
screening_radius = (
    1.5 * skyrmion_radius        # exclusion radius around each skyrmion (must be even or odd multiple)
)
skyrmion_smoothing = 1           # Gaussian smoothing applied to the skyrmion profile (pixels)

number_of_skyrmions = 60000      # target number of skyrmions to place
max_nr_iteration = 100000        # max placement attempts (brute-force packing — reduce if too slow)

magnetic_pattern_config = simulation_configuration.MagneticPatternConfig(
    pattern_type_method="skyrmion_pattern",
    shape=sample_shape[1:],
    real_space_pixel_size=real_space_pixel_size,
    pattern_config_length={
        "skyr_radius": skyrmion_radius,
        "screening_radius": screening_radius,
    },
    pattern_config={
        "number_skyr": number_of_skyrmions,
        "number_iter": max_nr_iteration,
        "sigma": skyrmion_smoothing,
    },
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_config.plot_pattern()

In [ ]:
magnetic_pattern = magnetic_pattern_config.magnetic_pattern  # 2-D scalar pattern, values in [-1, 1]

# Build a 3-D magnetization vector field (mx, my, mz) from the scalar out-of-plane pattern.
# mx = 0, my = sqrt(1 - mz²)  (in-plane component to preserve |m| = 1), mz = pattern
magnetization = pattern_generator.map_magnetization_to_3d(
    np.zeros_like(magnetic_pattern),                    # mx: no in-plane x-component
    np.sqrt(1 - np.abs(magnetic_pattern) ** 2),         # my: in-plane y-component (unit vector constraint)
    magnetic_pattern,                                    # mz: out-of-plane component
    nr_repeats=sample_shape[0],                          # repeat across all Nz layers
)

sampleconfig.assign_magnetic_pattern(magnetization)
metadata.update(magnetic_pattern_config.get_metadata(prefix="magnetic_pattern/"))  # record pattern params

The 2-D scalar pattern is promoted to a 3-D magnetization vector field `(Nz, Ny, Nx, 3)` with in-plane components set to zero and the out-of-plane component equal to the pattern. `Nz` is replicated across all layers — the same magnetic texture is assumed uniform through the magnetic layer stack.

# - holography mask

In [ ]:
# FTH aperture: one large object hole (OH) + two small reference holes (RH)
apertures_radius  = [120e-9, 8e-9, 7e-9]                          # hole radii in m
apertures_types   = ["OH", "RH", "RH"]                            # OH = object hole, RH = reference hole
apertures_centers = [(0, 0), (0.35e-6, -0.25e-6), (0.25e-6, 0.18e-6)]  # (y, x) centres in m
apertures_sigma   = [1e-9, 0.1e-9, 0.1e-9]                       # edge-smoothing sigma per hole in m

front_aperture_config = simulation_configuration.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=sample_shape,
    real_space_pixel_size=sampleconfig.sample_structure.real_space_pixel_size,
    aperture_thicknesses=sampleconfig.sample_structure.layer_thicknesses,  # per-layer thicknesses for RH depth sum
    aperture_config=dict(
        apertures_type=apertures_types,
        apertures_radius=apertures_radius,
        apertures_center=apertures_centers,
        apertures_sigma=apertures_sigma,
        thickness_OH=np.sum(
            sampleconfig.sample_structure.layer_thicknesses[
                : sampleconfig.sample_structure.layer_names.index("SiN")
            ]
        ),  # total depth of OH from surface down to the SiN membrane
    ),
)
front_aperture_config.setup()
front_aperture_config.visualize_aperture()

aperture_mask = front_aperture_config.return_aperture()
sampleconfig.assign_aperture_mask(aperture_mask)
metadata.update(front_aperture_config.get_metadata(prefix="aperture/"))  # record aperture geometry

In [ ]:
# Combine refractive indices and magnetization into a full 3×3 dielectric tensor per layer per pixel.
# Must be called after assign_magnetic_pattern() and assign_aperture_mask().
sampleconfig.sample_structure.calculate_final_dielectric_tensor()

The dielectric tensor encodes both the charge and magnetic contributions to the optical response of each layer. It is computed from the refractive indices (loaded for the given X-ray energy) and the magnetization vector field. This is the main input to the Jones propagator.

### HOLOGRAM COMPUTATION

In [ ]:
# Params for gaussian beam
illumination_function = "gaussian"
illumination_center = (0, 0)
illumination_focus_distance = 1e-3  # in m
illumination_fwhm = 0.5e-6  # in m, it is not showing correct fwhm?

illuminationconfig = simulation_configuration.IlluminationConfig(
    XRayConfig=xrayconfig,
    shape=sample_shape[-2:],
    real_space_pixel_size=real_space_pixel_size,
    illumination_function=illumination_function,
    illumination_config={
        "center": illumination_center,
        "distance": illumination_focus_distance,
        "fwhm": illumination_fwhm,
    },
)
illuminationconfig.setup()
illuminationconfig.visualize_illumination()

A Gaussian beam is used to mimic the focused synchrotron spot. `fwhm` controls the beam diameter at the focus; `distance` shifts the beam waist relative to the sample plane (0 = focus on sample). The beam is stored as a complex Jones wavefield.

In [ ]:
# Container that accumulates exit waves and holograms for all polarisations
hologram_config = simulation_configuration.HologramConfig(
    sample_x=sampleconfig.sample_structure.x,
    sample_y=sampleconfig.sample_structure.y,
    detector_layout=detectorconfig.detector_layout,
)

for i, polarization in enumerate(["CR", "CL"]):
    illuminationconfig.update_polarization(polarization)  # switch beam Jones vector to CR or CL

    samplepropagationconfig = simulation_configuration.SamplePropagatorConfig(
        SampleConfig=sampleconfig,
        IlluminationConfig=illuminationconfig,
        propagator_method="Jones",  # full Jones matrix propagation through the dielectric tensor
        propagator_config={},
    )
    samplepropagationconfig.setup()  # runs the propagation; result stored internally

    detectorconfig.assign_propagated_wavefront(samplepropagationconfig)  # project exit wave onto detector
    detectorconfig.detect_hologram()  # add Poisson shot noise for the "detected" version
    detectorconfig.hologram_exp.gnomonic_projection()  # correct for curved Ewald sphere geometry

    hologram_config.add_exit_waves(
        {polarization: samplepropagationconfig.return_scalar_wavefield()}
    )
    hologram_config.add_holograms(
        {polarization: detectorconfig.return_ideal_hologram()}, source="ideal"  # noise-free
    )
    hologram_config.add_holograms(
        {polarization: detectorconfig.return_detected_hologram()}, source="detected"  # Poisson noise
    )

# Metadata from the last loop iteration; detector and propagator params are polarisation-independent
metadata.update(detectorconfig.get_metadata(prefix="detector/"))
metadata.update(samplepropagationconfig.get_metadata())  # includes sample and illumination sub-metadata

The loop runs the full propagation pipeline for each polarisation (CR and CL):
- **Jones propagation** through the sample dielectric tensor → exit wavefield
- **Gnomonic projection** from the exit plane onto the curved detector geometry
- **Ideal hologram** — intensity on the detector without noise
- **Detected hologram** — Poisson shot noise added to mimic real detection

Results are accumulated in `hologram_config` keyed by helicity.

In [ ]:
# Quick sanity check: display frame-averaged exit wave amplitudes/phases and holograms for CR and CL
hologram_config.visualize_averages()

In [ ]:
hologram_config.compute_differences()    # CR - CL → magnetic contrast
hologram_config.compute_sums()           # CR + CL → charge background
hologram_config.compute_reconstructions() # FTH: fftshift(fft2(fftshift(holo))) for each key
hologram_config.compute_averages()       # average over frames if stacked

Post-processing steps:
- **difference** — CR − CL, isolating the magnetic contrast signal
- **sum** — CR + CL, giving the charge (non-magnetic) background
- **reconstructions** — FTH reconstruction via `fftshift(fft2(fftshift(holo)))` for each helicity
- **averages** — mean over frames if a multi-shot stack was simulated

In [ ]:
# Visualise the FTH reconstruction of the helicity difference from the ideal (noise-free) hologram
hologram_config.visualize_reconstruction(source="detected", helicity="diff")

# Export data

In [ ]:
# Save folder for generated data
output_folder = DATA_ROOT / "Data" / "simulation_name"  # DATA_ROOT is one level above the fomocid repo
data_fname = "simulation.h5"  # HDF5 file name

if not os.path.exists(output_folder):
    os.makedirs(output_folder)  # create the directory tree if it does not exist yet

In [ ]:
# Retrieve CR and CL arrays for exit wave, ideal hologram, and detected hologram
save_data = hologram_config.to_dict(
    helicities=["CR", "CL"], sources=["exit_wave", "ideal", "detected"]
)
# Flatten nested {helicity: {source: array}} into HDF5-compatible keys "helicity/source"
save_data = {
    f"{helicity}/{per_helicity_data}": data_array
    for helicity, nested_dict in save_data.items()
    for per_helicity_data, data_array in nested_dict.items()
}
save_data["beamstop_mask"] = (detectorconfig.detector_layout.beamstop,)  # 2-D boolean mask of the beamstop region

In [ ]:
utils.io.save_simulation_arrays_hdf5(
    join(output_folder, data_fname),   # full path to the output file
    arrays=save_data,                  # flattened dict: keys become HDF5 dataset paths
    metadata=metadata,                 # scalar params stored under the metadata/ group
    overwrite=True,                    # replace any existing file
)